# Bot-IoT Analysis and Classification (ver2)

This version improves reproducibility, class-imbalance evaluation, and model diagnostics while keeping the original notebook unchanged.

In [1]:
import warnings

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier

from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score, classification_report

warnings.filterwarnings("default")

## Dataset Download

https://unsw-my.sharepoint.com/personal/z5131399_ad_unsw_edu_au/_layouts/15/onedrive.aspx?id=%2Fpersonal%2Fz5131399%5Fad%5Funsw%5Fedu%5Fau%2FDocuments%2FBot%2DIoT%5FDataset%2FDataset%2F5%25&viewid=604d81f1%2D64a9%2D4a09%2D8464%2D3c45ff9ba8fe

In [2]:
RANDOM_STATE = 42
TRAIN_PATH = "UNSW_2018_IoT_Botnet_Final_10_best_Training.csv"
TEST_PATH = "UNSW_2018_IoT_Botnet_Final_10_best_Testing.csv"

FEATURE_COLUMNS = [
    "seq",
    "stddev",
    "N_IN_Conn_P_SrcIP",
    "min",
    "state_number",
    "mean",
    "N_IN_Conn_P_DstIP",
    "drate",
    "srate",
    "max",
]

TARGET_COLUMNS = ["attack", "category", "subcategory"]

In [3]:
train_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH)

print("Train shape:", train_df.shape)
print("Test shape :", test_df.shape)

FileNotFoundError: [Errno 2] No such file or directory: 'UNSW_2018_IoT_Botnet_Final_10_best_Training.csv'

In [ ]:
for col in TARGET_COLUMNS:
    print(f"\n{col} distribution (train):")
    print(train_df[col].value_counts(normalize=True).head(10))

In [ ]:
X_all = train_df[FEATURE_COLUMNS].copy()
y_all = train_df[TARGET_COLUMNS].copy()

X_test_official = test_df[FEATURE_COLUMNS].copy()
y_test_official = test_df[TARGET_COLUMNS].copy()

In [ ]:
label_encoders = {}

for target_col in ["category", "subcategory"]:
    le = LabelEncoder()
    y_all[target_col] = le.fit_transform(y_all[target_col])
    y_test_official[target_col] = le.transform(y_test_official[target_col])
    label_encoders[target_col] = le

In [ ]:
stratify_key = (
    y_all["attack"].astype(str) + "_" +
    y_all["category"].astype(str) + "_" +
    y_all["subcategory"].astype(str)
)

X_train, X_valid, y_train, y_valid = train_test_split(
    X_all,
    y_all,
    test_size=0.25,
    random_state=RANDOM_STATE,
    stratify=stratify_key
)

print("X_train:", X_train.shape, "X_valid:", X_valid.shape)

In [ ]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_valid_scaled = scaler.transform(X_valid)

X_all_scaled = scaler.fit_transform(X_all)
X_test_official_scaled = scaler.transform(X_test_official)

In [ ]:
class ChainedMultiTargetClassifier:
    def __init__(self, attack_model, category_model, subcategory_model):
        self.attack_model = attack_model
        self.category_model = category_model
        self.subcategory_model = subcategory_model

    def fit(self, X, y):
        self.attack_model.fit(X, y["attack"])

        category_features = np.concatenate([X, y[["attack"]].to_numpy()], axis=1)
        self.category_model.fit(category_features, y["category"])

        subcategory_features = np.concatenate([category_features, y[["category"]].to_numpy()], axis=1)
        self.subcategory_model.fit(subcategory_features, y["subcategory"])
        return self

    def predict(self, X):
        pred_attack = self.attack_model.predict(X)

        category_features = np.concatenate([X, pred_attack.reshape(-1, 1)], axis=1)
        pred_category = self.category_model.predict(category_features)

        subcategory_features = np.concatenate([category_features, pred_category.reshape(-1, 1)], axis=1)
        pred_subcategory = self.subcategory_model.predict(subcategory_features)

        return pd.DataFrame({
            "attack": pred_attack,
            "category": pred_category,
            "subcategory": pred_subcategory,
        })

In [ ]:
def make_models(seed=RANDOM_STATE):
    return {
        "RandomForest": ChainedMultiTargetClassifier(
            RandomForestClassifier(max_depth=5, n_estimators=200, random_state=seed, class_weight="balanced_subsample", n_jobs=-1),
            RandomForestClassifier(max_depth=7, n_estimators=200, random_state=seed, class_weight="balanced_subsample", n_jobs=-1),
            RandomForestClassifier(max_depth=9, n_estimators=200, random_state=seed, class_weight="balanced_subsample", n_jobs=-1),
        ),
        "NaiveBayes": ChainedMultiTargetClassifier(
            GaussianNB(),
            GaussianNB(),
            GaussianNB(),
        ),
        "DecisionTreeEntropy": ChainedMultiTargetClassifier(
            DecisionTreeClassifier(criterion="entropy", max_depth=8, class_weight="balanced", random_state=seed),
            DecisionTreeClassifier(criterion="entropy", max_depth=10, class_weight="balanced", random_state=seed),
            DecisionTreeClassifier(criterion="entropy", max_depth=12, class_weight="balanced", random_state=seed),
        ),
        "DecisionTreeGini": ChainedMultiTargetClassifier(
            DecisionTreeClassifier(criterion="gini", max_depth=8, class_weight="balanced", random_state=seed),
            DecisionTreeClassifier(criterion="gini", max_depth=10, class_weight="balanced", random_state=seed),
            DecisionTreeClassifier(criterion="gini", max_depth=12, class_weight="balanced", random_state=seed),
        ),
        "XGBoost": ChainedMultiTargetClassifier(
            XGBClassifier(
                n_estimators=300,
                max_depth=6,
                learning_rate=0.1,
                subsample=0.8,
                colsample_bytree=0.8,
                random_state=seed,
                eval_metric="logloss",
                n_jobs=-1
            ),
            XGBClassifier(
                n_estimators=300,
                max_depth=6,
                learning_rate=0.1,
                subsample=0.8,
                colsample_bytree=0.8,
                random_state=seed,
                eval_metric="mlogloss",
                n_jobs=-1
            ),
            XGBClassifier(
                n_estimators=300,
                max_depth=6,
                learning_rate=0.1,
                subsample=0.8,
                colsample_bytree=0.8,
                random_state=seed,
                eval_metric="mlogloss",
                n_jobs=-1
            ),
        ),
    }

In [ ]:
def evaluate_predictions(y_true, y_pred, target_col):
    y_t = y_true[target_col]
    y_p = y_pred[target_col]

    return {
        "target": target_col,
        "accuracy": accuracy_score(y_t, y_p),
        "balanced_accuracy": balanced_accuracy_score(y_t, y_p),
        "macro_f1": f1_score(y_t, y_p, average="macro", zero_division=0),
        "weighted_f1": f1_score(y_t, y_p, average="weighted", zero_division=0),
    }

def evaluate_model(name, model, X_eval, y_eval):
    pred = model.predict(X_eval)
    rows = []
    for target_col in TARGET_COLUMNS:
        metric_row = evaluate_predictions(y_eval, pred, target_col)
        metric_row["model"] = name
        rows.append(metric_row)
    return rows, pred

In [ ]:
validation_rows = []
validation_preds = {}

models = make_models()
for name, model in models.items():
    model.fit(X_train_scaled, y_train)
    rows, preds = evaluate_model(name, model, X_valid_scaled, y_valid)
    validation_rows.extend(rows)
    validation_preds[name] = preds

validation_result = pd.DataFrame(validation_rows).sort_values(["target", "macro_f1"], ascending=[True, False])
validation_result

In [ ]:
print("Detailed report (validation) for XGBoost / category:")
print(classification_report(y_valid["category"], validation_preds["XGBoost"]["category"], zero_division=0))

In [ ]:
test_rows = []

models_full = make_models()
for name, model in models_full.items():
    model.fit(X_all_scaled, y_all)
    rows, _ = evaluate_model(name, model, X_test_official_scaled, y_test_official)
    test_rows.extend(rows)

test_result = pd.DataFrame(test_rows).sort_values(["target", "macro_f1"], ascending=[True, False])
test_result

## Notes in ver2

- Reproducibility: fixed `RANDOM_STATE` and stratified split.
- Better imbalance visibility: added `balanced_accuracy` and `macro_f1`.
- Cleaner warnings behavior: no global warning suppression.
- XGBoost warning cleanup: explicit `eval_metric` values.